In [1]:
import subprocess
import os

# ── Forcer Java 21 depuis Python ──
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.11"
os.environ["PATH"] = r"C:\Program Files\Java\jdk-21.0.11\bin;" + os.environ["PATH"]

# ── Vérification ──
SPMF_JAR      = "../spmf.jar"
FICHIER_INPUT = "../outputs/sequences/sequences_spmf.txt"
FICHIER_OUTPUT = "../outputs/sequences/resultats_prefixspan_30.txt"
SUPPORT_MIN   = "0.3"

print("Vérification des fichiers...")
print(f"  spmf.jar        : {'✓' if os.path.exists(SPMF_JAR) else '✗ INTROUVABLE'}")
print(f"  sequences_spmf  : {'✓' if os.path.exists(FICHIER_INPUT) else '✗ INTROUVABLE'}")

result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(f"\nJava : {result.stderr.splitlines()[0] if result.stderr else '✗ Java non trouvé'}")

Vérification des fichiers...
  spmf.jar        : ✓
  sequences_spmf  : ✓

Java : java version "21.0.11" 2026-04-21 LTS


In [2]:
print("Lancement de PrefixSpan...")

result = subprocess.run(
    [
        "java", "-jar", SPMF_JAR,
        "run", "PrefixSpan",
        FICHIER_INPUT,
        FICHIER_OUTPUT,
        SUPPORT_MIN
    ],
    capture_output=True,
    text=True
)

print("STDOUT :", result.stdout)
print("STDERR :", result.stderr)

if os.path.exists(FICHIER_OUTPUT):
    print(f"\n✓ Résultats sauvegardés dans {FICHIER_OUTPUT}")
    with open(FICHIER_OUTPUT, "r") as f:
        lignes = f.readlines()
    print(f"  Patterns trouvés : {len(lignes):,}")
    print(f"\nAperçu des 10 premiers patterns :")
    for ligne in lignes[:10]:
        print(f"  {ligne.strip()}")
else:
    print("✗ Fichier de sortie non généré")

Lancement de PrefixSpan...
STDOUT : >/C:/Projects/GTD_Analyst/spmf.jar
=============  PREFIXSPAN - STATISTICS =============
Total time ~ 344
 Frequent sequences count : 779
 Max memory (mb) : 114.1334228515625
 minsup = 696 sequences.
 Pattern count : 779
 Minsup (as number of sequences)) : 696


STDERR : 

✓ Résultats sauvegardés dans ../outputs/sequences/resultats_prefixspan_30.txt
  Patterns trouvés : 779

Aperçu des 10 premiers patterns :
  1 -1 #SUP: 1419
  1 -1 1 -1 #SUP: 1337
  1 -1 1 -1 1 -1 #SUP: 1267
  1 -1 1 -1 1 -1 1 -1 #SUP: 1209
  1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1159
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1092
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1043
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 985
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 916
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 861


In [3]:
# ── Discrétisation ──
def discretiser(val):
    if val == 0:   return 0  # inactif
    elif val <= 2: return 1  # faible
    elif val <= 10: return 2  # modéré
    elif val <= 50: return 3  # élevé
    else:           return 4  # très élevé

FICHIER_INPUT_DISC  = "../outputs/sequences/sequences_spmf_disc_30.txt"
FICHIER_OUTPUT_DISC = "../outputs/sequences/resultats_prefixspan_disc_30.txt"

# Relire le fichier original et discrétiser
with open("../outputs/sequences/sequences_spmf.txt", "r") as f:
    lignes = f.readlines()

lignes_disc = []
for ligne in lignes:
    # Extraire les valeurs (tout sauf -1 et -2)
    tokens = ligne.strip().split()
    nouveaux_tokens = []
    for t in tokens:
        if t == "-2":
            nouveaux_tokens.append("-2")
        elif t == "-1":
            nouveaux_tokens.append("-1")
        else:
            nouveaux_tokens.append(str(discretiser(int(t))))
    lignes_disc.append(" ".join(nouveaux_tokens))

with open(FICHIER_INPUT_DISC, "w") as f:
    f.write("\n".join(lignes_disc))

print(f"Séquences discrétisées sauvegardées !")
print(f"  Fichier : {FICHIER_INPUT_DISC}")
print(f"  Lignes  : {len(lignes_disc):,}")

# Aperçu
print(f"\nAperçu avant/après discrétisation :")
for i in range(3):
    print(f"  Avant : {lignes[i].strip()[:60]}")
    print(f"  Après : {lignes_disc[i][:60]}")
    print()

Séquences discrétisées sauvegardées !
  Fichier : ../outputs/sequences/sequences_spmf_disc_30.txt
  Lignes  : 2,320

Aperçu avant/après discrétisation :
  Avant : 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 
  Après : 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 

  Avant : 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 
  Après : 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 

  Avant : 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 
  Après : 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 0 -1 



In [4]:
FICHIER_INPUT_DISC  = "../outputs/sequences/sequences_spmf_disc_30.txt"
FICHIER_OUTPUT_DISC = "../outputs/sequences/resultats_prefixspan_disc_30.txt"
SUPPORT_MIN_DISC    = "0.3"

print("Lancement de PrefixSpan sur séquences discrétisées...")

result = subprocess.run(
    [
        "java", "-jar", SPMF_JAR,
        "run", "PrefixSpan",
        FICHIER_INPUT_DISC,
        FICHIER_OUTPUT_DISC,
        SUPPORT_MIN_DISC
    ],
    capture_output=True,
    text=True
)

print("STDOUT :", result.stdout)

if os.path.exists(FICHIER_OUTPUT_DISC):
    with open(FICHIER_OUTPUT_DISC, "r") as f:
        lignes = f.readlines()
    print(f"\n✓ Patterns trouvés : {len(lignes):,}")
    print(f"\nAperçu des 15 premiers patterns :")
    for ligne in lignes[:15]:
        print(f"  {ligne.strip()}")
else:
    print("✗ Fichier de sortie non généré")

Lancement de PrefixSpan sur séquences discrétisées...
STDOUT : >/C:/Projects/GTD_Analyst/spmf.jar
=============  PREFIXSPAN - STATISTICS =============
Total time ~ 470
 Frequent sequences count : 1426
 Max memory (mb) : 102.7838363647461
 minsup = 696 sequences.
 Pattern count : 1426
 Minsup (as number of sequences)) : 696



✓ Patterns trouvés : 1,426

Aperçu des 15 premiers patterns :
  1 -1 #SUP: 1424
  1 -1 1 -1 #SUP: 1354
  1 -1 1 -1 1 -1 #SUP: 1297
  1 -1 1 -1 1 -1 1 -1 #SUP: 1248
  1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1203
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1159
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1133
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1095
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1054
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 1016
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 975
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 #SUP: 946
  1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1 1 -1

In [5]:
readme_content = f"""# Séquences GTD — Documentation

## Paramètres de génération
- Période         : {ANNEE_DEBUT_QT}–{ANNEE_FIN_QT}
- Fréquence       : {PERIODE_QT} (Q=trimestrielle, M=mensuelle, 6M=semestrielle)
- Seuil division  : {SEUIL_DIVISION} incidents (Quadtree)
- Profondeur max  : {PROFONDEUR_MAX}

## Fichiers

### sequences_spmf.txt
Format brut pour SPMF. Chaque ligne = une cellule.
Format : valeur -1 valeur -1 ... -2
- Chaque valeur = nb d'incidents bruts sur la période
- -1 = séparateur entre périodes
- -2 = fin de séquence
- Une ligne = une cellule quadtree dans l'ordre de qt["cell_id"]

### sequences_lisible.txt
Format humain. Chaque ligne = une cellule.
Format : cell_id | periode:nb_incidents, periode:nb_incidents, ...

### sequences_spmf_disc.txt
Même format que sequences_spmf.txt mais avec valeurs discrétisées :
- 0 = inactif      (0 incident)
- 1 = faible       (1–2 incidents)
- 2 = modéré       (3–10 incidents)
- 3 = élevé        (11–50 incidents)
- 4 = très élevé   (> 50 incidents)

### [cell_id].txt
Un fichier par cellule avec :
- Métadonnées de la cellule (bornes, profondeur, centre)
- Séquence au format SPMF
- Détail par période (période : nb_incidents)

## Correspondance cellule ↔ séquence
- L'ordre des lignes dans sequences_spmf.txt correspond à l'ordre des cell_id dans qt
- Chaque fichier individuel [cell_id].txt porte l'identifiant de la cellule
- La ligne N dans sequences_spmf.txt correspond à qt.iloc[N]["cell_id"]

## Discrétisation
| Niveau | Valeur | Nb incidents |
|--------|--------|--------------|
| Inactif     | 0 | 0            |
| Faible      | 1 | 1–2          |
| Modéré      | 2 | 3–10         |
| Élevé       | 3 | 11–50        |
| Très élevé  | 4 | > 50         |

## Algorithme SPMF utilisé
- Algorithme : PrefixSpan
- Support minimum : 30% ({int(len(qt) * 0.3):,} cellules minimum)
"""

with open("../outputs/sequences/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README sauvegardé dans outputs/sequences/README.md !")

NameError: name 'ANNEE_DEBUT_QT' is not defined